# Waste Type Identification — Generazione split per 5-fold Cross-Validation

Questo notebook prende lo split 80/20 gia esistente (`splits/split.csv`) e lo trasforma
in uno schema a **5 fold** (`splits/split_cv.csv`), aggiungendo una colonna `fold` con valori 1..5.

Risultato: 5 fold disgiunti, ognuno ~20% del dataset, che insieme coprono tutte le immagini.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, random
from pathlib import Path
from collections import defaultdict

BASE      = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
SPLIT_CSV = BASE / 'splits' / 'split.csv'                                       # Split 80/20 esistente
OUT_CSV   = BASE / 'splits' / 'split_cv.csv'                                    # Nuovo file con la colonna 'fold' (1..5)

K    = 5
SEED = 1234
random.seed(SEED)                                                               # Per la riproducibilità

Mounted at /content/drive


In [ ]:
df = pd.read_csv(SPLIT_CSV)
print(f"Letto {SPLIT_CSV}  ({len(df)} immagini)")

# Fold 1 = validation set attuale
df['fold'] = 0
df.loc[df['split'] == 'val', 'fold'] = 1

# Fold 2..5 = partizione stratificata del train attuale
train_idx = df.index[df['split'] == 'train'].tolist()

groups = defaultdict(list)
for i in train_idx:
    key = (df.at[i, 'macro_label'], df.at[i, 'sub_label'])
    groups[key].append(i)

remaining_folds = [2, 3, 4, 5]
for key, idxs in groups.items():
    random.shuffle(idxs)
    for j, i in enumerate(idxs):
        df.at[i, 'fold'] = remaining_folds[j % len(remaining_folds)]

Letto /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/splits/split.csv  (15515 immagini)


In [ ]:
# Controllo
assert (df['fold'] == 0).sum() == 0, "Ci sono righe senza fold assegnato!"

# Salvataggio
df_out = df[['filepath', 'label', 'macro_label', 'sub_label', 'fold']]
df_out.to_csv(OUT_CSV, index=False)
print(f"Salvato: {OUT_CSV}  (totale {len(df_out)} immagini, K={K} fold)\n")

# Riepilogo
print("Numero immagini per macro-classe x fold:")
pivot = df_out.pivot_table(index='macro_label', columns='fold',
                           values='filepath', aggfunc='count', fill_value=0)
print(pivot)

print("\nTotale immagini per fold:")
print(df_out['fold'].value_counts().sort_index())


Salvato: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/splits/split_cv.csv  (totale 15515 immagini, K=5 fold)

Numero immagini per macro-classe x fold:
fold                 1     2     3     4     5
macro_label                                   
battery            189   189   189   189   189
clothing          1460  1461  1461  1460  1460
glass              402   403   403   402   401
metal              154   154   154   154   153
organic            197   197   197   197   197
papery             388   389   388   388   388
plastic            173   173   173   173   173
undifferentiated   139   140   140   139   139

Totale immagini per fold:
fold
1    3102
2    3106
3    3105
4    3102
5    3100
Name: count, dtype: int64
